# End-to-End BYOC (Bring Your Own Chunks) with Visual Grounding

This notebook demonstrates the complete process for successfully ingesting custom chunks with associated images and retrieving them in Vertex AI Search.

**Process:**
1.  **Create a Vertex AI Search Data Store**: To hold our unstructured data.
2.  **Create a Search App (Engine)**: To provide the search and answer generation capabilities.
3.  **Ingest a BYOC Document**: The core step, showing how to map `blobAssets` (images) directly to specific chunks using `chunkFields`.
4.  **Query the App**: Use `streamAnswer` to retrieve both the custom chunk content and the associated image (`blobAttachments`).

In [ ]:
# %% [markdown]
# ### Step 0: Install Dependencies & Authenticate

# %%
!pip install google-cloud-discoveryengine -q

# %%
# If you are running this in Google Colab, uncomment the following lines to authenticate.
# from google.colab import auth
# auth.authenticate_user()

# %%
import time
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions

# --- CONFIGURATION ---
# Please replace these with your actual project details.
PROJECT_ID = "YOUR_PROJECT_ID"
LOCATION = "global"  # Or "us", "eu"
DATA_STORE_ID = f"byoc-image-guide-ds-{int(time.time())}"  # Unique ID for the Data Store
ENGINE_ID = f"byoc-image-guide-app-{int(time.time())}"     # Unique ID for the Search App

# Set up client options for the specified location
client_options = (
    ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com")
    if LOCATION != "global"
    else None
)

print(f"Using Project ID: {PROJECT_ID}")
print(f"Data Store ID: {DATA_STORE_ID}")
print(f"Engine ID: {ENGINE_ID}")


# %% [markdown]
# ### Step 1: Create the Data Store
#
# We will create an Unstructured Data Store designed for search solutions.

# %%
def create_data_store():
    """Creates a new data store."""
    ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    data_store = discoveryengine.DataStore(
        display_name="BYOC Image Demo Store",
        industry_vertical=discoveryengine.IndustryVertical.GENERIC,
        solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
        content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
    )

    print(f"Creating Data Store: {DATA_STORE_ID}...")
    try:
        request = discoveryengine.CreateDataStoreRequest(
            parent=parent,
            data_store_id=DATA_STORE_ID,
            data_store=data_store,
        )
        operation = ds_client.create_data_store(request=request)
        response = operation.result()
        print(f"Data Store created successfully: {response.name}")
        return response
    except Exception as e:
        print(f"Error creating data store: {e}")
        print("This may happen if the data store already exists. Please check the console.")


create_data_store()


# %% [markdown]
# ### Step 2: Create the Search App (Engine)
#
# Next, we create a Search App with LLM add-ons enabled (required for `streamAnswer`) and link it to our new Data Store.

# %%
def create_engine():
    """Creates a new search app."""
    engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
    parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

    engine = discoveryengine.Engine(
        display_name="BYOC Image Demo App",
        solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
        data_store_ids=[DATA_STORE_ID],
        search_engine_config=discoveryengine.Engine.SearchEngineConfig(
            search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
            search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM],
        ),
    )

    print(f"Creating Engine (App): {ENGINE_ID}...")
    try:
        request = discoveryengine.CreateEngineRequest(
            parent=parent,
            engine_id=ENGINE_ID,
            engine=engine,
        )
        operation = engine_client.create_engine(request=request)
        response = operation.result()
        print(f"Engine created successfully: {response.name}")
        return response
    except Exception as e:
        print(f"Error creating engine: {e}")
        print("This may happen if the engine already exists. Please check the console.")


create_engine()

# Wait for a brief period to ensure the backend is fully provisioned before importing documents.
print("\nWaiting 20 seconds for provisioning to complete...")
time.sleep(20)


# %% [markdown]
# ### Step 3: Ingest the BYOC Document with Image Mapping
#
# This is the most critical step. We will construct a document payload that contains:
# 1.  `blob_assets`: The raw base64-encoded image data.
# 2.  `chunked_document`: Our custom-defined chunks.
# 3.  **`chunk_fields`**: The explicit mapping that tells Vertex AI Search which image (`blobAssetId`) belongs to which chunk.

# %%
def ingest_byoc_document():
    """Ingests a BYOC document with an image explicitly mapped to a chunk."""
    doc_client = discoveryengine.DocumentServiceClient(client_options=client_options)
    parent = doc_client.branch_path(
        PROJECT_ID, LOCATION, DATA_STORE_ID, "default_branch"
    )

    # For demonstration, we use a tiny, transparent 1x1 PNG.
    # In a real scenario, you would load your image and base64-encode it.
    # e.g., with open("path/to/your/image.png", "rb") as f: encoded_image = base64.b64encode(f.read()).decode('utf-8')
    encoded_image = "iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAQAAAC1HAwCAAAAC0lEQVR42mNkYAAAAAYAAjCB0C8AAAAASUVORK5CYII="

    # 1. Define the BlobAsset that holds the image data
    blob_asset = discoveryengine.Document.BlobAsset(
        name="blob_1",  # This is the Blob Asset ID we will reference.
        content=encoded_image,
        mime_type="image/png",
    )

    # 2. **CRITICAL STEP**: Map the Blob Asset to a Chunk Field
    # This creates the link between the chunk and the image.
    image_chunk_field = discoveryengine.Document.ChunkedDocument.Chunk.ImageChunkField(
        blob_asset_id="blob_1"  # Reference the ID from the BlobAsset above.
    )
    chunk_field = discoveryengine.Document.ChunkedDocument.Chunk.ChunkField(
        name="chart_image",  # A descriptive name for this field.
        image_chunk_field=image_chunk_field,
    )

    # 3. Create the Custom Chunk and attach the Chunk Field
    chunk = discoveryengine.Document.ChunkedDocument.Chunk(
        chunk_id="custom_chunk_with_image_1",
        content="This chunk describes the price volatility of Ethylene, Caustic soda, and Benzene from 2000 to 2019.",
        # This list explicitly tells VAS that the image belongs to this chunk.
        chunk_fields=[chunk_field],
    )

    chunked_document = discoveryengine.Document.ChunkedDocument(chunks=[chunk])

    # 4. Assemble the final Document payload
    document = discoveryengine.Document(
        id="byoc_doc_with_mapped_image_001",
        blob_assets=[blob_asset],  # Include the image data at the document level.
        chunked_document=chunked_document, # Include the chunk data.
    )

    # 5. Import the Document
    print("Ingesting BYOC document...")
    request = discoveryengine.ImportDocumentsRequest(
        parent=parent,
        inline_source=discoveryengine.ImportDocumentsRequest.InlineSource(
            documents=[document]
        ),
    )

    operation = doc_client.import_documents(request=request)
    response = operation.result()
    print("Import operation initiated successfully!")
    print("Document indexing can take several minutes.")
    return response

ingest_byoc_document()

# Note: Indexing chunks can take 5-10 minutes.
# We will wait here before querying to increase the chance of the document being ready.
print("\nWaiting 5 minutes for indexing to complete...")
time.sleep(300)

# %% [markdown]
# ### Step 4: Query the App and Verify Blob Attachments
#
# Finally, we query the app using `streamAnswer` and inspect the response. We expect to see the `blobAttachments` field populated in the citation source because we correctly mapped the image to the chunk during ingestion.

# %%
def query_stream_answer():
    """Queries the app with streamAnswer and checks for blob attachments."""
    conv_client = discoveryengine.ConversationalSearchServiceClient(
        client_options=client_options
    )
    serving_config = conv_client.serving_config_path(
        PROJECT_ID, LOCATION, ENGINE_ID, "default_serving_config"
    )

    query_text = "price volatility of Ethylene, Caustic soda, and Benzene from 2000 to 2019"

    request = discoveryengine.AnswerQueryRequest(
        serving_config=serving_config,
        query=discoveryengine.Query(text=query_text),
        answer_generation_spec=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec(
            include_citations=True,
            multimodal_spec=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec.MultimodalSpec(
                image_source=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec.MultimodalSpec.ImageSource.CORPUS_IMAGE_ONLY
            ),
            model_spec=discoveryengine.AnswerQueryRequest.AnswerGenerationSpec.ModelSpec(
                model_version="stable"
            ),
        ),
    )

    print(f"\nSending streamAnswer query: '{query_text}'...")

    response_stream = conv_client.answer_query_stream(request=request)

    found_attachments = False
    for chunk in response_stream:
        # Check citations for the retrieved chunk information
        if chunk.answer.citations:
            for citation in chunk.answer.citations:
                for source in citation.sources:
                    print("\n--- Citation Source Found ---")
                    print(f"Custom Chunk Content: '{source.chunk_info.content}'")

                    # **Verification Step**
                    if source.chunk_info.blob_attachments:
                        found_attachments = True
                        print(
                            f"\n[SUCCESS] Found {len(source.chunk_info.blob_attachments)} Blob Attachment(s)!"
                        )
                        for blob in source.chunk_info.blob_attachments:
                            print(f" -> Blob Name: {blob.name}")
                            print(f" -> MIME Type: {blob.mime_type}")
                            print(f" -> Base64 Content Length: {len(blob.content)}")
                    else:
                        print("\n[WARNING] No Blob Attachments were returned in this source.")

    if not found_attachments:
        print("\n--- Final Result ---")
        print("Query completed, but no blob attachments were found.")
        print("This could be because indexing is not yet complete. Please try running this cell again in a few minutes.")

# Run the query
query_stream_answer()

